# Lie Groups and Symmetries

| Difficulty | ████████░░ 8/10 |
| :--- | :--- |
| Prerequisites | 10.03 |
| Time | ~85 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/10_Differential_Geometry_and_Topology/04_lie_groups_symmetries.ipynb)

---

## 🎯 Learning Objective

Learn basic group theory and Lie groups, and connect symmetry and equivariance to representation theory.

---

## 📐 Theory

Every previous notebook in this module studied a *specific* curved space (a manifold of points,
`10.01`; a manifold of distributions, `10.02`–`10.03`). This notebook studies spaces whose
points are themselves **transformations** — rotations, translations — and asks what structure
that extra ingredient (points that can be *composed*, not just moved between) adds.

### Groups: composable, invertible transformations

A **group** is a set $G$ with a composition rule ("multiplication") satisfying four axioms:
**closure** (composing two elements of $G$ gives another element of $G$), **associativity**
($(ab)c = a(bc)$), an **identity element** $e$ (composing with $e$ does nothing), and
**inverses** (every $g$ has a $g^{-1}$ with $gg^{-1}=e$). The rotations of the plane form a
group: rotating by $\theta_1$ then $\theta_2$ is the same as rotating by $\theta_1+\theta_2$
(closure), rotating by $0°$ does nothing (identity), and rotating backward undoes any rotation
(inverse). Crucially, groups need not be **abelian** ($ab=ba$ for all elements) — composing two
3D rotations in different orders can give different results, verified numerically below,
whereas 2D rotations always commute.

### Lie groups: groups that are also smooth manifolds

A **Lie group** is a group whose elements *also* form a smooth manifold (recall `10.01`), with
the group operations (composition, inversion) varying smoothly. $SO(2)$, the rotations of the
plane, is a Lie group: each rotation is indexed by one continuous angle $\theta\in[0,2\pi)$,
and this parameter space is itself a circle — a 1-dimensional manifold. $SO(3)$, the rotations
of 3D space, is a 3-dimensional Lie group (three numbers needed: an axis, given by 2 numbers on
a sphere, plus an angle).

### Lie algebras: the tangent space at the identity

Because a Lie group is a manifold, it has a tangent space at every point (`10.01`) — and the
tangent space at the *identity* element is special enough to have its own name: the **Lie
algebra** $\mathfrak{g}$. For $SO(2)$, the Lie algebra is spanned by a single generator matrix
$J = \begin{pmatrix}0&-1\\1&0\end{pmatrix}$, and every rotation can be recovered from it via the
matrix exponential: $R(\theta) = \exp(\theta J)$ — verified numerically below. This is the same
matrix exponential behind continuous-time linear ODEs (recall `06`'s numerical methods for
differential equations); a Lie algebra is, in a precise sense, the "infinitesimal" or
"linearized" version of its Lie group, and exponentiating recovers the full (curved, nonlinear)
group from that flat, linear tangent space.

### Group actions and equivariance

A group doesn't just exist abstractly — it **acts** on other spaces. $SO(2)$ acts on
$\mathbb{R}^2$ by rotating vectors: $g\cdot v = R(\theta)v$. A function $f$ is **equivariant**
with respect to a group action if applying the group action before or after $f$ gives the same
result: $f(g\cdot v) = g\cdot f(v)$ — transforming the input and then processing it gives the
same answer as processing it and then transforming the output. This is a strictly stronger,
more structured property than mere **invariance** ($f(g\cdot v) = f(v)$, where the group action
is discarded entirely rather than tracked through).

### Representation theory, briefly

**Representation theory** studies how an abstract group can *act* on vector spaces via matrices
— a **representation** is a map from group elements to matrices that respects the group's
composition rule ($\rho(g_1 g_2) = \rho(g_1)\rho(g_2)$). Every explicit matrix we write below
(each $R(\theta)$) is already a representation of $SO(2)$ or $SO(3)$ on $\mathbb{R}^2$ or
$\mathbb{R}^3$; the deeper theory (decomposing arbitrary representations into irreducible
pieces) is beyond this notebook's scope, but the core idea — a fixed abstract symmetry acting
consistently across many different concrete spaces — is exactly what motivates equivariant
neural network architectures below.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Applying every rotation in $SO(2)$ to a single fixed vector traces out its **orbit** — a circle
— and the tangent direction at each orbit point is generated by the Lie algebra element $J$,
made visible by plotting sampled tangent vectors along the orbit.

In [ ]:
def R2(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

J = np.array([[0.0, -1.0], [1.0, 0.0]])  # the Lie algebra generator of SO(2)
v0 = np.array([1.5, 0.5])

thetas = np.linspace(0, 2*np.pi, 200)
orbit = np.array([R2(t) @ v0 for t in thetas])

sample_thetas = np.linspace(0, 2*np.pi, 8, endpoint=False)
sample_points = np.array([R2(t) @ v0 for t in sample_thetas])
sample_tangents = np.array([J @ (R2(t) @ v0) for t in sample_thetas])  # d/dtheta [R(theta) v0]

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(orbit[:, 0], orbit[:, 1], color="#4C72B0", lw=2, label="orbit of $v_0$ under $SO(2)$")
ax.quiver(sample_points[:, 0], sample_points[:, 1], sample_tangents[:, 0], sample_tangents[:, 1],
          color="#C44E52", angles='xy', scale_units='xy', scale=1, width=0.006,
          label="tangent direction $J\\,v(\\theta)$")
ax.scatter(*v0, color="k", s=60, zorder=5, label="$v_0$")
ax.set_aspect('equal')
ax.set_title("The orbit of a vector under $SO(2)$ is a circle;\nthe Lie algebra generator $J$ gives its tangent direction everywhere")
ax.legend(loc='upper right')
plt.show()

radii = np.linalg.norm(orbit, axis=1)
print(f"All orbit points at the same radius (confirms the orbit is exactly a circle): "
      f"{np.allclose(radii, np.linalg.norm(v0))}")
tangent_dot_radius = [np.dot(p, tv) for p, tv in zip(sample_points, sample_tangents)]
print(f"Tangent vectors are perpendicular to the radius at every sample point "
      f"(max |dot product|={np.max(np.abs(tangent_dot_radius)):.2e}): "
      f"{np.allclose(tangent_dot_radius, 0, atol=1e-10)}")

## 🐍 Implementation from Scratch

We verify the group axioms (closure, identity, inverse, associativity) numerically for both
$SO(2)$ and $SO(3)$, confirm $SO(2)$ is abelian while $SO(3)$ is not (checked over many random
pairs, not just one), and verify the Lie algebra exponential map $\exp(\theta J) = R(\theta)$
recovers the group from its tangent space at the identity.

In [ ]:
from scipy.linalg import expm

rng = np.random.default_rng(0)

def R2(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

# --- SO(2) group axioms ---
theta1, theta2, theta3 = rng.uniform(0, 2*np.pi, size=3)
A, B, C = R2(theta1), R2(theta2), R2(theta3)

closure_ok = np.isclose(np.linalg.det(A @ B), 1.0) and np.allclose((A @ B) @ (A @ B).T, np.eye(2))
identity_ok = np.allclose(A @ np.eye(2), A) and np.allclose(np.eye(2) @ A, A)
inverse_ok = np.allclose(A @ A.T, np.eye(2))  # orthogonal matrices: inverse = transpose
assoc_ok = np.allclose((A @ B) @ C, A @ (B @ C))

print("SO(2) group axioms:")
print(f"  Closure (A@B is orthogonal, det=1):  {closure_ok}")
print(f"  Identity element acts trivially:     {identity_ok}")
print(f"  Inverse (R^T R = I):                 {inverse_ok}")
print(f"  Associativity ((AB)C = A(BC)):       {assoc_ok}")
print(f"  Abelian (AB = BA):                    {np.allclose(A @ B, B @ A)}")

# --- Lie algebra: the generator J and the exponential map ---
J = np.array([[0.0, -1.0], [1.0, 0.0]])
theta_test = 0.7
R_from_exp = expm(theta_test * J)
R_direct = R2(theta_test)
print(f"\nLie algebra exponential map exp(theta*J) reproduces R(theta) directly: "
      f"{np.allclose(R_from_exp, R_direct)}")
print("exp(theta J):\n", np.round(R_from_exp, 4))
print("R(theta) via cos/sin directly:\n", np.round(R_direct, 4))

# --- SO(3): axis-angle (Rodrigues' formula) rotation matrices ---
def random_rotation_3d(rng):
    axis = rng.normal(size=3); axis /= np.linalg.norm(axis)
    angle = rng.uniform(0, 2*np.pi)
    K = np.array([[0, -axis[2], axis[1]], [axis[2], 0, -axis[0]], [-axis[1], axis[0], 0]])
    return np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * (K @ K)

R1_3d, R2_3d, R3_3d = (random_rotation_3d(rng) for _ in range(3))
closure_3d = np.isclose(np.linalg.det(R1_3d @ R2_3d), 1.0) and np.allclose((R1_3d @ R2_3d) @ (R1_3d @ R2_3d).T, np.eye(3))
inverse_3d = np.allclose(R1_3d @ R1_3d.T, np.eye(3))
assoc_3d = np.allclose((R1_3d @ R2_3d) @ R3_3d, R1_3d @ (R2_3d @ R3_3d))

print(f"\nSO(3) group axioms:")
print(f"  Closure: {closure_3d}    Inverse: {inverse_3d}    Associativity: {assoc_3d}")

n_trials = 200
non_commuting = sum(
    1 for _ in range(n_trials)
    if not np.allclose((Ra := random_rotation_3d(rng)) @ (Rb := random_rotation_3d(rng)),
                        Rb @ Ra, atol=1e-8)
)
print(f"  Non-abelian check: {non_commuting}/{n_trials} random SO(3) pairs do NOT commute "
      f"({100*non_commuting/n_trials:.0f}%) -- unlike SO(2), which is always abelian.")

## 🤖 Why This Matters for AI

Convolutional neural networks are already an equivariant architecture, though this is rarely
stated in those terms: a convolution layer is **translation-equivariant** by construction —
shifting the input image shifts the output feature map by exactly the same amount, verified
numerically below. This isn't an accident; it's the entire reason convolutions generalize so
well for images, where "a cat shifted 10 pixels to the left is still a cat in the same way."
**Geometric deep learning** and **equivariant neural networks** generalize this idea
deliberately: instead of only building in translation symmetry, choose network layers that are
provably equivariant to whatever symmetry group is actually relevant to the data — rotations for
molecules and point clouds, permutations for graphs and sets, the full Euclidean group for 3D
structures like proteins (AlphaFold's core architectural bet). Below, we confirm translation
equivariance directly, then show that whether an ordinary 2D convolution is *also*
rotation-equivariant depends entirely on the shape of its kernel — an isotropic (rotationally
symmetric) kernel nearly commutes with rotation, while a directional kernel does not, which is
exactly why architectures that need genuine rotation equivariance must be built from group
theory rather than assumed for free.

In [ ]:
from scipy.ndimage import convolve, rotate

rng_ai = np.random.default_rng(0)

def conv2d(image, kern):
    return convolve(image, kern, mode='constant')

# --- Translation equivariance: shift-then-convolve vs convolve-then-shift ---
img = rng_ai.normal(size=(40, 40))
kernel = rng_ai.normal(size=(5, 5))

conv_out = conv2d(img, kernel)
shift = (3, -2)
img_shifted = np.roll(img, shift, axis=(0, 1))
conv_of_shifted = conv2d(img_shifted, kernel)
shifted_conv_output = np.roll(conv_out, shift, axis=(0, 1))

# np.roll wraps around at the border, but convolution zero-pads -- compare only the interior,
# far enough from the border that the wrap-around discrepancy can't reach.
interior = (slice(10, 30), slice(10, 30))
match = np.allclose(conv_of_shifted[interior], shifted_conv_output[interior], atol=1e-8)
print(f"Translation equivariance of an ordinary convolution (interior region): {match}")
print(f"conv(shift(x)) == shift(conv(x)) exactly, away from the zero-padded border -- this")
print(f"IS the mathematical statement that a conv layer is translation-equivariant.")

# --- Rotation equivariance: does an ordinary conv2d commute with rotation? Depends on the kernel. ---
img2 = np.zeros((41, 41))
img2[15:26, 18:23] = 1.0  # an off-center rectangle, so rotation is a genuine test
kernel_iso = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]], dtype=float)      # isotropic (rotationally symmetric)
kernel_aniso = np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]], dtype=float)  # directional (Sobel-like), NOT isotropic

def rotate_img(image, angle):
    return rotate(image, angle, reshape=False, order=1, mode='constant')

angle = 37.0

def relative_commutator_error(kern):
    conv_then_rotate = rotate_img(conv2d(img2, kern), angle)
    rotate_then_conv = conv2d(rotate_img(img2, angle), kern)
    return np.linalg.norm(conv_then_rotate - rotate_then_conv) / np.linalg.norm(conv_then_rotate)

diff_iso = relative_commutator_error(kernel_iso)
diff_aniso = relative_commutator_error(kernel_aniso)

print(f"\nRelative difference between conv-then-rotate and rotate-then-conv:")
print(f"  Isotropic (rotation-symmetric) kernel:  {diff_iso:.4f}")
print(f"  Anisotropic (directional) kernel:       {diff_aniso:.4f}")
print(f"Isotropic kernel is far closer to rotation-equivariant: {diff_iso < diff_aniso}")

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
axes[0].imshow(img2, cmap="gray"); axes[0].set_title("input")
axes[1].imshow(rotate_img(conv2d(img2, kernel_iso), angle), cmap="gray")
axes[1].set_title(f"isotropic kernel:\nrotate(conv(x)), diff={diff_iso:.3f}")
axes[2].imshow(rotate_img(conv2d(img2, kernel_aniso), angle), cmap="gray")
axes[2].set_title(f"anisotropic kernel:\nrotate(conv(x)), diff={diff_aniso:.3f}")
plt.tight_layout()
plt.show()

print(f"\nOrdinary CNNs get translation equivariance for free from the convolution operation")
print(f"itself, but only get rotation equivariance 'for free' if every kernel happens to be")
print(f"isotropic -- which throws away most of a kernel's expressive power (an isotropic kernel")
print(f"can't detect edges at a specific orientation). Genuine rotation-equivariant architectures")
print(f"(e.g. steerable CNNs, spherical CNNs) instead constrain kernels using representation")
print(f"theory of the rotation group, keeping expressive power while still guaranteeing equivariance.")

## 🏋️ Exercises

### Warm-up
1. Verify by hand that $J^2 = -I$ for the $SO(2)$ generator matrix $J$ used above (this is why
   $\exp(\theta J) = \cos(\theta) I + \sin(\theta) J$ — the same identity that makes $J$ behave
   like the imaginary unit $i$ under exponentiation). Confirm with `J @ J`.

### Practice
2. Pick two *fixed* (not random) 3D rotations, one a $90°$ rotation about the x-axis and one a
   $90°$ rotation about the y-axis (construct each using `random_rotation_3d`'s formula with a
   fixed axis and `angle=np.pi/2`). Compute both orderings of their product and print the
   resulting rotation matrices. Confirm they differ, and describe in words (or by tracking where
   a specific point like $(0,0,1)$ ends up) what physically different rotation each ordering
   produces.

### Challenge
3. In the AI section, `kernel_iso` is a fixed $3\times3$ isotropic kernel. Construct a *larger*
   isotropic kernel (e.g. $7\times7$, with values depending only on distance from the center —
   a discretized radial function) and re-run `relative_commutator_error`. Does a larger
   isotropic kernel's rotation-equivariance error get better or worse compared to the $3\times3$
   case? Explain the result in terms of what "isotropic" can only approximately mean on a
   discrete pixel grid (a true continuous rotation has no exact discrete-grid equivalent).

---

## 📚 Further Reading
- Artin, *Algebra*, Ch. 2 and 6 (Groups, Symmetry)
- Cohen & Welling, [\"Group Equivariant Convolutional Networks\"](https://arxiv.org/abs/1602.07576)
- Bronstein et al., [\"Geometric Deep Learning: Going beyond Euclidean data\"](https://arxiv.org/abs/1611.08097)

---

*Next notebook: [Topological Data Analysis](05_topological_data_analysis.ipynb)*